# 2. Data Preprocessing

This notebook handles audio preprocessing, feature extraction, and dataset creation for the BirdClef 2026 competition.

In [1]:
import pandas as pd
import numpy as np
import os
import torch
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

BASE_DIR = Path('/home/kwierman/Desktop/Projects/BirdClef')
TRAIN_AUDIO_DIR = BASE_DIR / 'train_audio'
TRAIN_SOUNDSCAPES_DIR = BASE_DIR / 'train_soundscapes'
PROCESSED_DIR = BASE_DIR / 'notebooks' / 'data' / 'processed'
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

# Load metadata
train_df = pd.read_csv(BASE_DIR / 'train.csv')
taxonomy_df = pd.read_csv(BASE_DIR / 'taxonomy.csv')
train_soundscapes_labels = pd.read_csv(BASE_DIR / 'train_soundscapes_labels.csv')

print("Loaded datasets successfully")

Loaded datasets successfully


## 2.1 Create Label Mappings

In [2]:
# Create species to index mapping
species_list = taxonomy_df['primary_label'].tolist()
species_to_idx = {species: idx for idx, species in enumerate(species_list)}
idx_to_species = {idx: species for species, idx in species_to_idx.items()}

num_classes = len(species_list)
print(f"Number of classes: {num_classes}")
print(f"\nFirst 10 species: {species_list[:10]}")
print(f"Last 10 species: {species_list[-10:]}")

# Save mappings
np.save(PROCESSED_DIR / 'species_to_idx.npy', species_to_idx)
np.save(PROCESSED_DIR / 'idx_to_species.npy', idx_to_species)
print("\nLabel mappings saved")

Number of classes: 234

First 10 species: ['1161364', '116570', '1176823', '1491113', '1595929', '209233', '22930', '22956', '22961', '22967']
Last 10 species: ['whnjay1', 'whtdov', 'whwpic1', 'y00678', 'yebcar', 'yebela1', 'yecmac', 'yecpar', 'yehcar1', 'yeofly1']

Label mappings saved


## 2.2 Audio Loading Utilities

In [3]:
# Check available audio libraries
try:
    import torchaudio
    print("Using torchaudio for audio processing")
    TORCHAUDIO_AVAILABLE = True
except ImportError:
    print("torchaudio not available")
    TORCHAUDIO_AVAILABLE = False

try:
    import librosa
    print("librosa is available")
    LIBROSA_AVAILABLE = True
except ImportError:
    print("librosa not available")
    LIBROSA_AVAILABLE = False

print(f"\nAudio processing options: torchaudio={TORCHAUDIO_AVAILABLE}, librosa={LIBROSA_AVAILABLE}")

Using torchaudio for audio processing
librosa is available

Audio processing options: torchaudio=True, librosa=True


In [4]:
def load_audio(file_path, target_sr=32000):
    """Load audio file and resample to target sample rate."""
    if TORCHAUDIO_AVAILABLE:
        waveform, sr = torchaudio.load(file_path)
        if sr != target_sr:
            resampler = torchaudio.transforms.Resample(sr, target_sr)
            waveform = resampler(waveform)
        return waveform, target_sr
    else:
        # Fallback: use scipy to read (won't work for ogg directly)
        raise RuntimeError("Audio loading requires torchaudio")

def get_audio_duration(file_path):
    """Get duration of audio file in seconds."""
    if TORCHAUDIO_AVAILABLE:
        info = torchaudio.info(str(file_path))
        return info.num_frames / info.sample_rate
    return None

## 2.3 Feature Extraction: Mel Spectrogram

In [5]:
class AudioFeatureExtractor:
    """Extract mel spectrogram features from audio."""
    
    def __init__(self, sample_rate=32000, n_fft=2048, hop_length=512, n_mels=128):
        self.sample_rate = sample_rate
        self.n_fft = n_fft
        self.hop_length = hop_length
        self.n_mels = n_mels
        
        if TORCHAUDIO_AVAILABLE:
            self.mel_spec = torchaudio.transforms.MelSpectrogram(
                sample_rate=sample_rate,
                n_fft=n_fft,
                hop_length=hop_length,
                n_mels=n_mels
            )
            self.amplitude_to_db = torchaudio.transforms.AmplitudeToDB()
    
    def extract(self, waveform):
        """Extract mel spectrogram from waveform tensor."""
        if TORCHAUDIO_AVAILABLE:
            mel = self.mel_spec(waveform)
            mel_db = self.amplitude_to_db(mel)
            return mel_db
        return None
    
    def extract_from_file(self, file_path):
        """Load and extract features from file."""
        waveform, sr = load_audio(file_path, self.sample_rate)
        return self.extract(waveform)

# Test feature extraction
extractor = AudioFeatureExtractor()
print("AudioFeatureExtractor initialized")

AudioFeatureExtractor initialized


## 2.4 Prepare Training Data from train_audio

In [6]:
def create_train_audio_dataset(df, audio_dir, max_samples_per_species=50):
    """Create a balanced dataset from train_audio."""
    
    # Sample max_samples_per_species from each species
    sampled_df = df.groupby('primary_label').apply(
        lambda x: x.sample(n=min(len(x), max_samples_per_species), random_state=42)
    ).reset_index(drop=True)
    
    # Create dataset entries
    dataset_entries = []
    
    for idx, row in sampled_df.iterrows():
        species = row['primary_label']
        filename = row['filename']
        
        # Construct file path
        file_path = audio_dir / filename
        
        if file_path.exists():
            dataset_entries.append({
                'file_path': str(file_path),
                'species': species,
                'label_idx': species_to_idx[species]
            })
    
    return pd.DataFrame(dataset_entries)

print("Creating training dataset from train_audio...")
train_audio_dataset = create_train_audio_dataset(train_df, TRAIN_AUDIO_DIR, max_samples_per_species=50)
print(f"Created dataset with {len(train_audio_dataset)} samples")
print(f"\nSamples per species (first 10):")
print(train_audio_dataset['species'].value_counts().head(10))

Creating training dataset from train_audio...


KeyError: 'primary_label'

## 2.5 Prepare Train Soundscapes Dataset

In [ ]:
def create_soundscapes_dataset(labels_df, audio_dir):
    """Create dataset from train_soundscapes with labels."""
    
    dataset_entries = []
    
    for filename in labels_df['filename'].unique():
        file_path = audio_dir / filename
        if file_path.exists():
            file_labels = labels_df[labels_df['filename'] == filename]
            
            for _, row in file_labels.iterrows():
                # Parse start time to get segment index
                start_time = row['start']
                segment_idx = int(start_time.split(':')[-1]) // 5  # 5-second segments
                
                # Parse labels
                labels = row['primary_label'].split(';')
                label_indices = [species_to_idx.get(l, -1) for l in labels if l in species_to_idx]
                
                dataset_entries.append({
                    'file_path': str(file_path),
                    'segment_idx': segment_idx,
                    'labels': labels,
                    'label_indices': label_indices
                })
    
    return pd.DataFrame(dataset_entries)

print("Creating soundscapes dataset...")
soundscapes_dataset = create_soundscapes_dataset(train_soundscapes_labels, TRAIN_SOUNDSCAPES_DIR)
print(f"Created soundscapes dataset with {len(soundscapes_dataset)} segment entries")
print(f"\nSample entry:")
print(soundscapes_dataset.iloc[0])

## 2.6 PyTorch Dataset Classes

In [ ]:
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn

class BirdDataset(Dataset):
    """Dataset for bird audio classification."""
    
    def __init__(self, dataframe, extractor, max_duration=5.0, augment=None):
        self.dataframe = dataframe.reset_index(drop=True)
        self.extractor = extractor
        self.max_duration = max_duration
        self.augment = augment
    
    def __len__(self):
        return len(self.dataframe)
    
    def __getitem__(self, idx):
        row = self.dataframe.iloc[idx]
        
        try:
            waveform, sr = load_audio(row['file_path'], target_sr=self.extractor.sample_rate)
            
            # Trim or pad to max_duration
            max_frames = int(self.max_duration * sr)
            if waveform.shape[1] > max_frames:
                waveform = waveform[:, :max_frames]
            elif waveform.shape[1] < max_frames:
                padding = max_frames - waveform.shape[1]
                waveform = torch.nn.functional.pad(waveform, (0, padding))
            
            # Extract features
            features = self.extractor.extract(waveform)
            
            # Handle multi-channel
            if features.shape[0] > 1:
                features = features.mean(dim=0, keepdim=True)
            
            # Get label
            label = torch.zeros(num_classes)
            if 'label_idx' in row:
                label[row['label_idx']] = 1.0
            elif 'label_indices' in row:
                for li in row['label_indices']:
                    if li >= 0:
                        label[li] = 1.0
            
            return features, label
            
        except Exception as e:
            # Return empty features on error
            features = torch.zeros(1, self.extractor.n_mels, int(self.max_duration * self.extractor.sample_rate // self.extractor.hop_length + 1))
            label = torch.zeros(num_classes)
            return features, label

print("BirdDataset class defined")

## 2.7 Create Multi-Label Targets

In [ ]:
# Create binary label matrix for train soundscapes
def create_multilabel_matrix(labels_df, species_to_idx, num_classes):
    """Create multi-label binary matrix."""
    labels_matrix = np.zeros((len(labels_df), num_classes), dtype=np.float32)
    
    for idx, labels in enumerate(labels_df['primary_label']):
        species_list = labels.split(';')
        for species in species_list:
            if species in species_to_idx:
                labels_matrix[idx, species_to_idx[species]] = 1.0
    
    return labels_matrix

labels_matrix = create_multilabel_matrix(train_soundscapes_labels, species_to_idx, num_classes)
print(f"Label matrix shape: {labels_matrix.shape}")
print(f"Label matrix dtype: {labels_matrix.dtype}")
print(f"\nPositive labels per sample (stats):")
print(f"  Mean: {labels_matrix.sum(axis=1).mean():.2f}")
print(f"  Max: {labels_matrix.sum(axis=1).max():.0f}")
print(f"  Min: {labels_matrix.sum(axis=1).min():.0f}")

# Save label matrix
np.save(PROCESSED_DIR / 'train_soundscapes_labels.npy', labels_matrix)
print("\nLabel matrix saved")

## 2.8 Data Augmentation

In [ ]:
class AudioAugmentation:
    """Audio augmentation for training."""
    
    def __init__(self, sample_rate=32000):
        self.sample_rate = sample_rate
    
    def time_stretch(self, waveform, rate=1.0):
        """Time stretch the audio."""
        if TORCHAUDIO_AVAILABLE:
            return torchaudio.functional.time_stretch(waveform, hop_length=512, rate=rate)
        return waveform
    
    def pitch_shift(self, waveform, n_steps=0):
        """Pitch shift the audio."""
        if TORCHAUDIO_AVAILABLE and n_steps != 0:
            return torchaudio.functional.phase_vocoder(waveform, hop_length=512, rate=2**(n_steps/12))
        return waveform
    
    def add_noise(self, waveform, noise_level=0.005):
        """Add random noise to audio."""
        noise = torch.randn_like(waveform) * noise_level
        return waveform + noise
    
    def time_shift(self, waveform, shift_max=0.2):
        """Randomly shift audio in time."""
        shift = int(torch.randint(0, int(waveform.shape[1] * shift_max), (1,)).item())
        return torch.roll(waveform, shifts=shift, dims=1)

print("AudioAugmentation class defined")

## 2.9 Create Train/Validation Split

In [ ]:
from sklearn.model_selection import train_test_split

# Filter species with at least 2 samples for stratification
species_counts = train_audio_dataset['species'].value_counts()
valid_species = species_counts[species_counts >= 2].index
train_audio_filtered = train_audio_dataset[train_audio_dataset['species'].isin(valid_species)].copy()

print(f"Filtered dataset: {len(train_audio_filtered)} samples ({len(valid_species)} species with >=2 samples)")
print(f"Removed {len(train_audio_dataset) - len(train_audio_filtered)} samples from {len(species_counts) - len(valid_species)} rare species")

# Split train_audio_filtered into train and validation
train_data, val_data = train_test_split(
    train_audio_filtered, 
    test_size=0.2, 
    random_state=42,
    stratify=train_audio_filtered['species']
)

print(f"Training samples: {len(train_data)}")
print(f"Validation samples: {len(val_data)}")

# Save splits
train_data.to_csv(PROCESSED_DIR / 'train_split.csv', index=False)
val_data.to_csv(PROCESSED_DIR / 'val_split.csv', index=False)
print("\nTrain/Validation splits saved")

## 2.10 Summary Statistics

In [ ]:
print("=" * 60)
print("DATA PREPROCESSING SUMMARY")
print("=" * 60)
print(f"\nNumber of classes: {num_classes}")
print(f"Train audio samples: {len(train_audio_dataset)}")
print(f"Train/Val split: {len(train_data)} / {len(val_data)}")
print(f"Train soundscapes segments: {len(soundscapes_dataset)}")
print(f"\nFeature extractor settings:")
print(f"  Sample rate: 32000 Hz")
print(f"  N FFT: 2048")
print(f"  Hop length: 512")
print(f"  N mel bands: 128")
print("\n✅ Data preprocessing complete!")